In [ ]:
pip install gym

In [9]:
pip install torch

   ---------------------------------------- 0.0/204.2 MB ? eta -:--:--
   ---------------------------------------- 0.4/204.2 MB 8.3 MB/s eta 0:00:25
   ---------------------------------------- 1.0/204.2 MB 10.7 MB/s eta 0:00:20
   ---------------------------------------- 1.9/204.2 MB 13.7 MB/s eta 0:00:15
    --------------------------------------- 2.8/204.2 MB 15.0 MB/s eta 0:00:14
    --------------------------------------- 3.2/204.2 MB 13.5 MB/s eta 0:00:15
    --------------------------------------- 3.5/204.2 MB 12.5 MB/s eta 0:00:17
    --------------------------------------- 3.9/204.2 MB 11.8 MB/s eta 0:00:17
    --------------------------------------- 4.2/204.2 MB 11.2 MB/s eta 0:00:18
    --------------------------------------- 4.5/204.2 MB 10.7 MB/s eta 0:00:19
    --------------------------------------- 4.9/204.2 MB 10.4 MB/s eta 0:00:20
   - -------------------------------------- 5.2/204.2 MB 10.1 MB/s eta 0:00:20
   - -------------------------------------- 5.6/204.2 MB 10.1

In [2]:
import gym
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
import glob
import io
import base64
from IPython.display import HTML
import IPython.display as ipythondisplay
import gym.wrappers

In [3]:
# Define Q-Network class
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

In [4]:
# Define Deep Q-Learning Agent
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.99  # Discount factor
        self.epsilon = 1.0  # Exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        
        self.model = QNetwork(state_size, action_size)
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.criterion = nn.MSELoss()
    
    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        state = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            action_values = self.model(state)
        return np.argmax(action_values.cpu().data.numpy())
    
    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))
    
    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)
        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                target += self.gamma * torch.max(self.model(torch.FloatTensor(next_state).unsqueeze(0))).item()
            target_f = self.model(torch.FloatTensor(state).unsqueeze(0))
            target_f[0][action] = target
            self.optimizer.zero_grad()
            loss = self.criterion(self.model(torch.FloatTensor(state).unsqueeze(0)), target_f)
            loss.backward()
            self.optimizer.step()
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

In [5]:
env = gym.make('CartPole-v1')

In [6]:
print(env.action_space)

Discrete(2)


In [7]:
print(env.observation_space)

Box([-4.8000002e+00 -3.4028235e+38 -4.1887903e-01 -3.4028235e+38], [4.8000002e+00 3.4028235e+38 4.1887903e-01 3.4028235e+38], (4,), float32)


In [8]:
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
agent = DQNAgent(state_size, action_size)

Task 1

In [9]:
state = env.reset()[0]
print("Sample State:", state)
print("Chosen Action:", agent.act(state))

Sample State: [ 0.01801933  0.0480601  -0.03990496  0.0267    ]
Chosen Action: 1


Task 2

In [10]:
episodes = 100
batch_size = 32
total_rewards = []
episode_numbers = []

for e in range(episodes):
    state = env.reset()[0]
    total_reward = 0
    for time in range(500):
        action = agent.act(state)
        next_state, reward, done, _, _ = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        if done:
            break
    agent.replay(batch_size)
    total_rewards.append(total_reward)

C:\Users\phuar\anaconda3\Lib\site-packages\gym\utils\passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


In [ ]:
# Plot cumulative rewards
plt.figure(figsize=(10, 5))
plt.plot(total_rewards, label="Reward per Episode")
plt.xlabel("Episode")
plt.ylabel("Cumulative Reward")
plt.title("Performance over 100 episodes")
plt.legend()
plt.grid()
plt.show()

print("Average Reward over 100 episodes:", np.mean(total_rewards))